# 타이타닉 41 — 전처리 & 모델링

> **대상**: 타이타닉 EDA(40)를 마친 단계.
> **목표**: EDA에서 찾은 특성으로 **결측치 처리 → 인코딩 → Pipeline → 모델 학습**까지, 32에서 배운 전처리를 실전 데이터에 적용하기.

40에서 "성별·등급이 생존을 가른다"는 걸 찾았고, "Age·Cabin·Embarked에 결측치가 있다"는 것도 봤습니다. 이제 그 데이터를 **모델이 먹을 수 있게 다듬어** 학습시킵니다. 핵심 도구는 32에서 본 `Pipeline`과, 수치/범주를 따로 처리하는 **`ColumnTransformer`**입니다.

## 0) 데이터 로드 & 특성 선택

```python
import pandas as pd
from pathlib import Path

DATA_DIR = Path.cwd().parent.parent / "data" / "titanic"
train = pd.read_csv(DATA_DIR / "train.csv")

# 쓸 특성: 수치형 / 범주형 구분 (전처리가 다르므로)
num_features = ["Age", "Fare", "SibSp", "Parch"]      # 수치 → 결측 채움 + 스케일
cat_features = ["Pclass", "Sex", "Embarked"]          # 범주 → 결측 채움 + 원핫

X = train[num_features + cat_features]
y = train["Survived"]                                  # 정답
```

> **특성 선택**: `Name`·`Ticket`·`Cabin`·`PassengerId`는 뺐습니다. `Cabin`은 결측 77%라 버리고, `Name`/`Ticket`은 그대로는 쓰기 어렵습니다(나중에 가공 가능). 40의 EDA가 "Pclass·Sex가 핵심"이라고 알려줬으니 그것들을 중심으로.

> 수치형과 범주형을 **나눈 이유**: 둘은 전처리가 다릅니다 — 수치는 "결측을 중앙값으로 채우고 스케일링", 범주는 "결측을 최빈값으로 채우고 원-핫". 이걸 한 번에 처리하는 게 `ColumnTransformer`(3번).

In [2]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path.cwd().parent.parent / "data" / "titanic"
train = pd.read_csv(DATA_DIR / "train.csv")

num_features = ["Age", "Fare", "SibSp", "Parch"]
cat_features = ["Pclass", "Sex", "Embarked"]

X = train[num_features + cat_features]
y = train["Survived"]

In [4]:
pd.DataFrame(X)

,Age,Fare,SibSp,Parch,Pclass,Sex,Embarked
0,22.0,7.2500,1,0,3,male,S
1,38.0,71.2833,1,0,1,female,C
2,26.0,7.9250,0,0,3,female,S
3,35.0,53.1000,1,0,1,female,S
4,35.0,8.0500,0,0,3,male,S
...,...,...,...,...,...,...,...
886,27.0,13.0000,0,0,2,male,S
887,19.0,30.0000,0,0,1,female,S
888,NaN,23.4500,1,2,3,female,S
889,26.0,30.0000,0,0,1,male,C


In [5]:
pd.DataFrame(y)

,Survived
0,0
1,1
2,1
3,1
4,0
...,...
886,0
887,1
888,0
889,1


## 1) 결측치 채우기 — SimpleImputer

40에서 본 결측치(Age 177, Embarked 2)를 채웁니다. 모델은 빈 값(NaN)을 못 먹으니까요.

```python
from sklearn.impute import SimpleImputer

# 수치형: 중앙값(median)으로 채움 — 이상치에 강함
num_imputer = SimpleImputer(strategy="median")
# Age의 중앙값은 28.0 → 빈 Age 177개가 모두 28.0으로 채워짐

# 범주형: 최빈값(most_frequent)으로 채움
cat_imputer = SimpleImputer(strategy="most_frequent")
# Embarked의 최빈값은 'S' → 빈 Embarked 2개가 'S'로
```

> **왜 median인가**: 평균(mean)은 이상치(아주 비싼 Fare 등)에 흔들리지만 중앙값은 안정적입니다. 나이·운임 같은 데이터엔 median이 무난해요. 범주형은 "가장 흔한 값"으로 채우는 게 자연스럽습니다.

In [9]:
from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy="median")

num_imputer

,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",False
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False


In [11]:
cat_imputer = SimpleImputer(strategy="most_frequent")
cat_imputer

,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'most_frequent'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",False
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False


In [19]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

In [20]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ("num", num_pipe, num_features),
    ("cat", cat_pipe, cat_features),
])

In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)),
])

model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3I

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['Age','Fare','SibSp',...,'Pclass','Sex','Embarked']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining column

In [23]:
model.score(X_test, y_test)

0.8044692737430168

## 대표 예제 — 두 모델 비교 (교차검증)

> 로지스틱 회귀와 랜덤포레스트를 같은 전처리로 학습하고, 5-fold 교차검증으로 비교하라.

```python
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

DATA_DIR = Path.cwd().parent.parent / "data" / "titanic"
train = pd.read_csv(DATA_DIR / "train.csv")

num_features = ["Age", "Fare", "SibSp", "Parch"]
cat_features = ["Pclass", "Sex", "Embarked"]
X, y = train[num_features + cat_features], train["Survived"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num_features),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_features),
])

for name, clf in [("로지스틱", LogisticRegression(max_iter=1000)),
                  ("랜덤포레스트", RandomForestClassifier(random_state=42))]:
    pipe = Pipeline([("pre", preprocessor), ("clf", clf)])
    cv = cross_val_score(pipe, X, y, cv=5).mean()
    print(f"{name}: {cv:.4f}")
```

**기대 결과:**

```
로지스틱: 0.7901
랜덤포레스트: 0.8059
```

> 가르칠 포인트: **같은 전처리, 모델만 교체**(30의 인터페이스 일관성). 랜덤포레스트가 살짝 높네요(0.81 vs 0.79). 교차검증으로 비교했으니 "한 번 운 좋은" 결과가 아닙니다. 타이타닉 베이스라인으로 80% 정도면 무난한 출발점이에요. (캐글 상위권은 특성 공학으로 83%+를 노립니다 — 42에서 개선)

In [25]:
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

DATA_DIR = Path.cwd().parent.parent / "data" / "titanic"
train = pd.read_csv(DATA_DIR / "train.csv")

num_features = ["Age", "Fare", "SibSp", "Parch"]
cat_features = ["Pclass", "Sex", "Embarked"]
X, y = train[num_features + cat_features], train["Survived"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                       ("sc", StandardScaler())]), num_features),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_features),
])

for name, clf in [("로지스틱", LogisticRegression(max_iter=1000)),
                  ("랜덤포레스트", RandomForestClassifier(random_state=42))]:
    pipe = Pipeline([("pre", preprocessor), ("clf", clf)])
    cv = cross_val_score(pipe, X, y, cv=5).mean()
    print(f"{name}: {cv:.4f}")
    

로지스틱: 0.7901
랜덤포레스트: 0.8059


## 검증 (노트북에서 실행)

```python
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score

DATA_DIR = Path.cwd().parent.parent / "data" / "titanic"
train = pd.read_csv(DATA_DIR / "train.csv")
num_f = ["Age", "Fare", "SibSp", "Parch"]
cat_f = ["Pclass", "Sex", "Embarked"]
X, y = train[num_f + cat_f], train["Survived"]

# Age 중앙값 = 28.0
assert SimpleImputer(strategy="median").fit(train[["Age"]]).statistics_[0] == 28.0

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num_f),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_f),
])
model = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])

# test 정확도 ≈ 0.80
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model.fit(Xtr, ytr)
assert abs(model.score(Xte, yte) - 0.8045) < 0.001

# CV 평균 ≈ 0.79
assert abs(cross_val_score(model, X, y, cv=5).mean() - 0.7901) < 0.001
print("통과 ✅")
```

In [27]:
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score

DATA_DIR = Path.cwd().parent.parent / "data" / "titanic"
train = pd.read_csv(DATA_DIR / "train.csv")
num_f = ["Age", "Fare", "SibSp", "Parch"]
cat_f = ["Pclass", "Sex", "Embarked"]
X, y = train[num_f + cat_f], train["Survived"]

# Age 중앙값 = 28.0
assert SimpleImputer(strategy="median").fit(train[["Age"]]).statistics_[0] == 28.0

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num_f),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_f),
])
model = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])

# test 정확도 ≈ 0.80
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model.fit(Xtr, ytr)
assert abs(model.score(Xte, yte) - 0.8045) < 0.001

# CV 평균 ≈ 0.79
assert abs(cross_val_score(model, X, y, cv=5).mean() - 0.7901) < 0.001
print("통과 ✅")

통과 ✅


## 직접 풀어보기 (연습)

`TODO`를 채운 뒤 검증 셀을 실행하세요.

### 연습 1 — Age 중앙값 채우기

> SimpleImputer로 Age를 중앙값으로 채울 때, 그 중앙값을 반환하라.
> **힌트**: `SimpleImputer(strategy="median").fit(df[["Age"]]).statistics_[0]`.

```python
from sklearn.impute import SimpleImputer

def solution(train):
    # TODO: Age 중앙값 반환
    pass

assert solution(train) == 28.0
print("통과 ✅")
```

### 연습 2 — 핵심 특성만으로 모델 (Sex + Pclass)

> Sex와 Pclass **두 특성만** 원-핫 인코딩해 로지스틱 회귀로 5-fold 교차검증 평균을 구하라. (round 4자리)
> **힌트**: `ColumnTransformer([("cat", OneHotEncoder(), ["Sex","Pclass"])])` + Pipeline + `cross_val_score`.

```python
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

def solution(train):
    X = train[["Sex", "Pclass"]]
    y = train["Survived"]
    # TODO: 원핫 → 로지스틱 → cross_val_score(cv=5) 평균 round 4
    pass

# 단 2개 특성만으로도 0.79 — EDA가 찾은 핵심 특성의 힘!
assert abs(solution(train) - 0.7867) < 0.001
print("통과 ✅")
```

### 연습 3 — 모델 바꿔 정확도 (도전)

> 전체 전처리 + `RandomForestClassifier(random_state=42)`로 test 정확도를 구하라. (`test_size=0.2, random_state=42, stratify`)

```python
from sklearn.ensemble import RandomForestClassifier

def solution(train, preprocessor):
    X = train[["Age","Fare","SibSp","Parch","Pclass","Sex","Embarked"]]
    y = train["Survived"]
    # TODO: Pipeline(preprocessor + RF) → 분할 → fit → test 정확도 round 4
    pass

# preprocessor는 위에서 만든 것 재사용
assert abs(solution(train, pre) - 0.8156) < 0.001
print("통과 ✅")
```